# Comparacao Sistematica de Modelos GARCH

Neste notebook, realizamos uma **comparacao sistematica** de todos os modelos GARCH
univariados disponiveis na biblioteca archbox.

**Modelos comparados:**
1. GARCH(1,1) - Bollerslev (1986)
2. EGARCH(1,1) - Nelson (1991)
3. GJR-GARCH(1,1) - Glosten, Jagannathan e Runkle (1993)
4. APARCH(1,1) - Ding, Granger e Engle (1993)
5. IGARCH(1,1) - Engle e Bollerslev (1986)
6. Component-GARCH - Engle e Lee (1999)

**Criterios de avaliacao:**
- Criterios de informacao (AIC, BIC, HQIC)
- Diagnosticos dos residuos (ARCH-LM, Ljung-Box)
- Previsao out-of-sample (rolling window)

**Conteudo:**
1. Estimando todos os modelos
2. Criterios de informacao
3. Diagnosticos dos residuos
4. Previsao out-of-sample
5. Ranking final
6. Conclusoes e recomendacoes praticas

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Estimando todos os modelos

Vamos estimar os 6 modelos GARCH univariados no dataset do **S&P 500**.

Cada modelo captura aspectos diferentes da dinamica da volatilidade:

| Modelo | Caracteristica principal |
|--------|-------------------------|
| GARCH | Baseline simetrico |
| EGARCH | Assimetria via $\ln(\sigma^2)$ |
| GJR-GARCH | Assimetria via indicadora |
| APARCH | Potencia flexivel + assimetria |
| IGARCH | Persistencia unitaria |
| CGARCH | Decomposicao curto/longo prazo |

In [ ]:
# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns'].values

# TODO: Estime todos os 6 modelos no dataset SP500
# Dicas:
# - Crie um dicionario para armazenar os resultados:
#   models = {}
#   results = {}
#
# - GARCH:    models['GARCH'] = GARCH(returns, p=1, q=1)
# - EGARCH:   models['EGARCH'] = EGARCH(returns, p=1, q=1)
# - GJR:      models['GJR'] = GJRGARCH(returns, p=1, q=1)
# - APARCH:   models['APARCH'] = APARCH(returns, p=1, q=1)
# - IGARCH:   models['IGARCH'] = IGARCH(returns)
# - CGARCH:   models['CGARCH'] = ComponentGARCH(returns)
#
# - Ajuste todos: results[name] = models[name].fit() para cada modelo
# - Imprima o summary de cada um

## 2. Criterios de informacao

Os criterios de informacao balanceiam **ajuste** (log-verossimilhanca) com **parcimonia** (numero de parametros):

$$\text{AIC} = -2\ell + 2k$$
$$\text{BIC} = -2\ell + k \ln(n)$$
$$\text{HQIC} = -2\ell + 2k \ln(\ln(n))$$

O BIC penaliza mais fortemente modelos com muitos parametros, sendo mais conservador
na selecao de modelos. Em amostras grandes ($n > 100$), $\ln(n) > 2$, entao BIC > AIC.

**Regra**: menor valor = melhor modelo.

In [ ]:
# TODO: Crie tabela comparativa com AIC/BIC/HQIC de cada modelo
# Dicas:
# - ic_table = pd.DataFrame({
#       name: {'AIC': res.aic, 'BIC': res.bic, 'HQIC': res.hqic,
#              'LogLik': res.loglike, 'Params': len(res.params)}
#       for name, res in results.items()
#   }).T
# - Ordene por AIC: ic_table.sort_values('AIC')
# - Use plot_model_comparison(results) para visualizar
# - Identifique o melhor modelo por cada criterio

## 3. Diagnosticos dos residuos

Um modelo bem especificado deve produzir **residuos padronizados** ($z_t = \epsilon_t / \sigma_t$)
que sejam i.i.d. Em particular:

- **Teste ARCH-LM** nos residuos: se p-valor > 0.05, o modelo capturou os efeitos ARCH
- **Ljung-Box** nos residuos ao quadrado ($z_t^2$): testa se ha autocorrelacao remanescente

Se os residuos passam nesses testes, o modelo esta **adequadamente especificado**.
Se falham, ha dependencia nao capturada — considere um modelo mais complexo.

In [ ]:
# TODO: Teste ARCH-LM e Ljung-Box nos residuos de cada modelo
# Dicas:
# - Para cada modelo:
#   resids = results[name].resid
#   arch_test = arch_lm_test(resids, lags=10)
#   lb_test = ljung_box_squared(resids, lags=10)
#
# - Crie tabela com estatisticas e p-valores de cada teste
# - Marque 'PASS' se p-valor > 0.05, 'FAIL' caso contrario
# - Qual(is) modelo(s) passam em todos os testes?

## 4. Previsao out-of-sample

Criterios de informacao avaliam o ajuste **in-sample**. Para avaliar a capacidade
**preditiva**, usamos a estrategia de **rolling window** (janela deslizante):

1. Defina uma janela de estimacao (ex: 1000 obs) e um horizonte de previsao (ex: 1 dia)
2. Estime o modelo na janela $[t-W+1, t]$
3. Faca previsao de $\sigma^2_{t+1}$
4. Compare com o proxy de variancia realizada: $r_{t+1}^2$
5. Deslize a janela 1 dia e repita

**Metrica**: RMSE (Root Mean Squared Error) da previsao:

$$\text{RMSE} = \sqrt{\frac{1}{T_{oos}} \sum_{t=1}^{T_{oos}} (\hat{\sigma}^2_{t+1} - r_{t+1}^2)^2}$$

In [ ]:
# TODO: Implemente rolling window e calcule RMSE da previsao
# Dicas:
# - window_size = 1000
# - n_forecasts = len(returns) - window_size
# - Para cada modelo e cada janela:
#   model = GARCH(returns[t-window_size:t], p=1, q=1)
#   res = model.fit(disp=False)
#   forecast = res.forecast(horizon=1)
#   predicted_var = forecast['variance'][0]
#   actual_var = returns[t] ** 2  # proxy de variancia realizada
#
# - Calcule RMSE para cada modelo
# - Use um subset de janelas para economizar tempo (ex: step=10)
# - Compare os RMSEs em uma tabela

## 5. Ranking final

Vamos combinar todos os criterios para um **ranking final** dos modelos:

1. **AIC** (ajuste in-sample com penalizacao leve)
2. **BIC** (ajuste in-sample com penalizacao forte)
3. **ARCH-LM p-valor** (adequacao dos residuos)
4. **RMSE out-of-sample** (capacidade preditiva)

Para cada criterio, atribuimos um **rank** (1 = melhor, 6 = pior).
O ranking final e a **media dos ranks**.

In [ ]:
# TODO: Crie ranking final combinando criterios in-sample e out-of-sample
# Dicas:
# - Crie DataFrame com todos os criterios
# - Para AIC/BIC/RMSE: menor e melhor -> rank ascendente
# - Para p-valor ARCH-LM: maior e melhor -> rank descendente
# - rank_table = ic_table[['AIC', 'BIC']].rank()
# - rank_table['RMSE_rank'] = rmse_series.rank()
# - rank_table['mean_rank'] = rank_table.mean(axis=1)
# - Ordene pelo rank medio e apresente o resultado

## 6. Conclusoes e recomendacoes praticas

### Guia de selecao de modelos GARCH

| Situacao | Modelo recomendado | Justificativa |
|----------|-------------------|---------------|
| **Baseline simples** | GARCH(1,1) | Robusto, poucos parametros, facil de interpretar |
| **Efeito alavancagem evidente** | GJR-GARCH ou EGARCH | Capturam assimetria com 1 parametro extra |
| **Potencia otima desconhecida** | APARCH | Deixa os dados determinarem $\delta$ |
| **Persistencia muito alta** | IGARCH | Formaliza $\alpha + \beta = 1$ |
| **Tendencia na volatilidade** | Component-GARCH | Separa curto e longo prazo |
| **Amostra pequena** | GARCH(1,1) | Evita sobreparametrizacao |

### Recomendacoes gerais

1. **Sempre comece pelo GARCH(1,1)** — e o benchmark universal
2. **Teste efeito alavancagem** com o Sign Bias test antes de adotar modelos assimetricos
3. **Use BIC para selecao** se a amostra e grande (BIC e consistente)
4. **Valide com diagnosticos** — o melhor AIC nao garante residuos limpos
5. **Avalie out-of-sample** quando o objetivo e previsao